<img src="https://s8.hostingkartinok.com/uploads/images/2018/08/308b49fcfbc619d629fe4604bceb67ac.jpg" width=500, height=450>
<h3 style="text-align: center;"><b>Физтех-Школа Прикладной математики и информатики (ФПМИ) МФТИ</b></h3>

---

***Some parts of the notebook are almost the copy of [ mmta-team course](https://github.com/mmta-team/mmta_fall_2020). Special thanks to mmta-team for making them publicly available. [Original notebook](https://github.com/mmta-team/mmta_fall_2020/blob/master/tasks/01_word_embeddings/task_word_embeddings.ipynb).***

<b> Прочитайте семинар, пожалуйста, для успешного выполнения домашнего задания. В конце ноутка напишите свой вывод. Работа без вывода оценивается ниже.

## Задача поиска схожих по смыслу предложений

Мы будем ранжировать вопросы [StackOverflow](https://stackoverflow.com) на основе семантического векторного представления

До этого в курсе не было речи про задачу ранжировния, поэтому введем математическую формулировку

## Задача ранжирования(Learning to Rank)

* $X$ - множество объектов
* $X^l = \{x_1, x_2, ..., x_l\}$ - обучающая выборка
<br>На обучающей выборке задан порядок между некоторыми элементами, то есть нам известно, что некий объект выборки более релевантный для нас, чем другой:
* $i \prec j$ - порядок пары индексов объектов на выборке $X^l$ c индексами $i$ и $j$
### Задача:
построить ранжирующую функцию $a$ : $X \rightarrow R$ такую, что
$$i \prec j \Rightarrow a(x_i) < a(x_j)$$

<img src="https://d25skit2l41vkl.cloudfront.net/wp-content/uploads/2016/12/Featured-Image.jpg" width=500, height=450>

### Embeddings

Будем использовать предобученные векторные представления слов на постах Stack Overflow.<br>
[A word2vec model trained on Stack Overflow posts](https://github.com/vefstathiou/SO_word2vec)

In [ ]:
!wget https://zenodo.org/record/1199620/files/SO_vectors_200.bin?download=1

--2026-03-18 06:04:04--  https://zenodo.org/record/1199620/files/SO_vectors_200.bin?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 188.185.43.153, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/1199620/files/SO_vectors_200.bin [following]
--2026-03-18 06:04:04--  https://zenodo.org/records/1199620/files/SO_vectors_200.bin
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1453905423 (1.4G) [application/octet-stream]
Saving to: ‘SO_vectors_200.bin?download=1’

SO_vectors_200.bin? 100%[===================>]   1.35G  23.5MB/s    in 68s     

2026-03-18 06:05:13 (20.4 MB/s) - ‘SO_vectors_200.bin?download=1’ saved [1453905423/1453905423]



In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 103.7 MB/s eta 0:00:00


In [ ]:
from gensim.models.keyedvectors import KeyedVectors
wv_embeddings = KeyedVectors.load_word2vec_format("SO_vectors_200.bin?download=1", binary=True)

#### Как пользоваться этими векторами?

Посмотрим на примере одного слова, что из себя представляет embedding

In [ ]:
word = 'dog'
if word in wv_embeddings:
    print(wv_embeddings[word].dtype, wv_embeddings[word].shape)

float32 (200,)


In [ ]:
print(f"Num of words: {len(wv_embeddings.index_to_key)}")

Num of words: 1787145


Найдем наиболее близкие слова к слову `dog`:

#### ***Вопрос 1:***
* Входит ли слово `cat` в топ-5 близких слов к слову `dog`? Какое место оно занимает?


In [ ]:
# method most_simmilar
wv_embeddings.most_similar(positive=['dog'], topn=5)

[('animal', 0.8564180135726929),
 ('dogs', 0.7880866527557373),
 ('mammal', 0.7623804211616516),
 ('cats', 0.7621253728866577),
 ('animals', 0.760793924331665)]

In [ ]:
wv_embeddings.most_similar(positive=['dog'], topn=30)

[('animal', 0.8564180135726929),
 ('dogs', 0.7880866527557373),
 ('mammal', 0.7623804211616516),
 ('cats', 0.7621253728866577),
 ('animals', 0.760793924331665),
 ('feline', 0.7392398715019226),
 ('bird', 0.7315488457679749),
 ('animal1', 0.7219215631484985),
 ('doggy', 0.7213349342346191),
 ('labrador', 0.7209131717681885),
 ('canine', 0.7209057211875916),
 ('meow', 0.7185295820236206),
 ('cow', 0.7080444693565369),
 ('dog2', 0.7057910561561584),
 ('woof', 0.705061137676239),
 ('dog1', 0.7038840651512146),
 ('dog3', 0.701882004737854),
 ('penguin', 0.697029173374176),
 ('bulldog', 0.6940488815307617),
 ('mammals', 0.6931389570236206),
 ('bark', 0.6913798451423645),
 ('fruit', 0.6892251968383789),
 ('reptile', 0.6891209483146667),
 ('furry', 0.6863499283790588),
 ('carnivore', 0.6862949728965759),
 ('cat', 0.6852341294288635),
 ('horse', 0.6833381652832031),
 ('kitten', 0.6820152997970581),
 ('sheep', 0.6802570223808289),
 ('chihuahua', 0.6791757941246033)]

***Ваш ответ:*** 'cat' не входит в топ-5 ближайших слов к 'dog' *(но 'cats' входит)*

Оноо находится на 26м месте (даже слово fruit выше)

### Векторные представления текста

Перейдем от векторных представлений отдельных слов к векторным представлениям вопросов, как к **среднему** векторов всех слов в вопросе. Если для какого-то слова нет предобученного вектора, то его нужно пропустить. Если вопрос не содержит ни одного известного слова, то нужно вернуть нулевой вектор.

In [ ]:
import numpy as np
import re
# you can use your tokenizer
# for example, nltk.tokenize import WordPunctTokenizer
from nltk.tokenize import WordPunctTokenizer
class MyTokenizer:
    def __init__(self):
        pass
    def tokenize(self, text):
        return re.findall('\w+', text)
tokenizer = MyTokenizer()

<>:10: SyntaxWarning: invalid escape sequence '\w'
<>:10: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_4207/1150731098.py:10: SyntaxWarning: invalid escape sequence '\w'
  return re.findall('\w+', text)


In [ ]:
def question_to_vec(question, embeddings, tokenizer, dim=200):
    """
        question: строка
        embeddings: наше векторное представление
        dim: размер любого вектора в нашем представлении

        return: векторное представление для вопроса
    """

    words = tokenizer.tokenize(question)
    embeds = [embeddings[word] for word in words if word in embeddings]
    if embeds == []:
        return np.zeros(dim)
    return np.mean(embeds, axis=0)

Теперь у нас есть метод для создания векторного представления любого предложения.

#### ***Вопрос 2:***

* Какая третья (с индексом 2) компонента вектора предложения `I love neural networks` (округлите до 2 знаков после запятой)?

In [ ]:
# Предложение
question = "I love neural networks"

tokenizer = WordPunctTokenizer()
round(question_to_vec(question, wv_embeddings, tokenizer)[2], 2)

np.float32(-1.29)

In [ ]:
# example of non-existing word
question_to_vec("xxx11213xx", wv_embeddings, tokenizer)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

### Оценка близости текстов

Представим, что мы используем идеальные векторные представления слов. Тогда косинусное расстояние между дублирующими предложениями должно быть меньше, чем между случайно взятыми предложениями.

Сгенерируем для каждого из $N$ вопросов $R$ случайных отрицательных примеров и примешаем к ним также настоящие дубликаты. Для каждого вопроса будем ранжировать с помощью нашей модели $R + 1$ примеров и смотреть на позицию дубликата. Мы хотим, чтобы дубликат был первым в ранжированном списке.

#### Hits@K
Первой простой метрикой будет количество корректных попаданий для какого-то $K$:
$$ \text{Hits@K} = \frac{1}{N}\sum_{i=1}^N \, [rank\_q_i^{'} \le K],$$
* $\begin{equation*}
[x < 0 ] \equiv
 \begin{cases}
   1, &x < 0\\
   0, &x \geq 0
 \end{cases}
\end{equation*}$ - индикаторная функция
* $q_i$ - $i$-ый вопрос
* $q_i^{'}$ - его дубликат
* $rank\_q_i^{'}$ - позиция дубликата в ранжированном списке ближайших предложений для вопроса $q_i$.

Hits@K  измеряет долю вопросов, для которых правильный ответ попал в топ-K позиций среди отранжированных кандидатов.

#### DCG@K
Второй метрикой будет упрощенная DCG метрика, учитывающая порядок элементов в списке путем домножения релевантности элемента на вес равный обратному логарифму номера позиции::
$$ \text{DCG@K} = \frac{1}{N} \sum_{i=1}^N\frac{1}{\log_2(1+rank\_q_i^{'})}\cdot[rank\_q_i^{'} \le K],$$
С такой метрикой модель штрафуется за большой ранк корректного ответа.

DCG@K  измеряет качество ранжирования, учитывая не только факт наличия правильного ответа в топ-K, но и ***его точную позицию***.

<img src='https://hsto.org/files/1c5/edf/dee/1c5edfdeebce4b71a86bdf986d9f88f2.jpg' width=400, height=200>

#### Пример оценок

Вычислим описанные выше метрики для игрушечного примера.
Пусть
* $N = 1$, $R = 3$
* <font color='green'>"Что такое python?"</font> - вопрос $q_1$
* <font color='red'>"Что такое язык python?"</font> - его дубликат $q_i^{'}$

Пусть модель выдала следующий ранжированный список кандидатов:

1. "Как изучить с++?"
2. <font color='red'>"Что такое язык python?"</font>
3. "Хочу учить Java"
4. "Не понимаю Tensorflow"

$\Rightarrow rank\_q_i^{'} = 2$

Вычислим метрику *Hits@K* для *K = 1, 4*:

- [K = 1] $\text{Hits@1} =  [rank\_q_i^{'} \le 1]$

Проверяем условие $ \text{rank}_{q'_1} \leq 1 $: ***условие неверно***.

Следовательно, $[\text{rank}_{q'_1} \leq 1] = 0$.

- [K = 4] $\text{Hits@4} =  [rank\_q_i^{'} \le 4] = 1$

Проверяем условие $ \text{rank}_{q'_1} \leq 4 $: ***условие верно***.

Вычислим метрику *DCG@K* для *K = 1, 4*:
- [K = 1] $\text{DCG@1} = \frac{1}{\log_2(1+2)}\cdot[2 \le 1] = 0$
- [K = 4] $\text{DCG@4} = \frac{1}{\log_2(1+2)}\cdot[2 \le 4] = \frac{1}{\log_2{3}}$

#### ***Вопрос 3***:
* Вычислите `DCG@10`, если $rank\_q_i^{'} = 9$(округлите до одного знака после запятой)



$\text{DCG@10} = \frac{1}{\log_2(1+9)}\cdot[9 \le 10] = \frac{1}{\log_2(10)} = 0.3$


#### Более сложный пример оценок

Рассмотрим пример с $ N > 1 $, где $ N = 3 $ (три вопроса) и для каждого вопроса заданы позиции их дубликатов. Вычислим метрики **Hits@K** для разных значений $ K $.

---

- $ N = 3 $: Три вопроса ($ q_1, q_2, q_3 $).
- Для каждого вопроса известна позиция его дубликата ($ \text{rank}_{q'_i} $):
  - $ \text{rank}_{q'_1} = 2 $,
  - $ \text{rank}_{q'_2} = 5 $,
  - $ \text{rank}_{q'_3} = 1 $.

Мы будем вычислять **Hits@K** для $ K = 1, 5 $.

---

**Для $ K = 1 $:**

Подставим значения:
$$
\text{Hits@1} = \frac{1}{3} \cdot \left( [\text{rank}_{q'_1} \leq 1] + [\text{rank}_{q'_2} \leq 1] + [\text{rank}_{q'_3} \leq 1] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 1 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 1 $ → $ 1 $.

Сумма:
$$
\text{Hits@1} = \frac{1}{3} \cdot (0 + 0 + 1) = \frac{1}{3}.
$$

$$
\boxed{\text{Hits@1} = \frac{1}{3}}.
$$

---

**Для $ K = 5 $:**

Подставим значения:
$$
\text{Hits@5} = \frac{1}{3} \cdot \left( [\text{rank}_{q'_1} \leq 5] + [\text{rank}_{q'_2} \leq 5] + [\text{rank}_{q'_3} \leq 5] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 5 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 5 $ → $ 1 $.

Сумма:
$$
\text{Hits@5} = \frac{1}{3} \cdot (1 + 1 + 1) = 1.
$$

$$
\boxed{\text{Hits@5} = 1}.
$$

Теперь вычислим метрику **DCG@K** для того же примера, где $ N = 3 $ (три вопроса), и для каждого вопроса известна позиция его дубликата ($ \text{rank}_{q'_i} $):

- $ \text{rank}_{q'_1} = 2 $,
- $ \text{rank}_{q'_2} = 5 $,
- $ \text{rank}_{q'_3} = 1 $.

Мы будем вычислять **DCG@K** для $ K = 1, 5 $.

---
**Для $ K = 1 $:**
Подставим значения:
$$
\text{DCG@1} = \frac{1}{3} \cdot \left( \frac{1}{\log_2(1 + \text{rank}_{q'_1})} \cdot [\text{rank}_{q'_1} \leq 1] + \frac{1}{\log_2(1 + \text{rank}_{q'_2})} \cdot [\text{rank}_{q'_2} \leq 1] + \frac{1}{\log_2(1 + \text{rank}_{q'_3})} \cdot [\text{rank}_{q'_3} \leq 1] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 1 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 1 $ → $ 1 $.

Сумма:
$$
\text{DCG@1} = \frac{1}{3} \cdot (0 + 0 + 1) = \frac{1}{3}.
$$
$$
\boxed{\text{DCG@1} = \frac{1}{3}}.
$$

---


**Для $ K = 5 $:**
Подставим значения:
$$
\text{DCG@5} = \frac{1}{3} \cdot \left( \frac{1}{\log_2(1 + \text{rank}_{q'_1})} \cdot [\text{rank}_{q'_1} \leq 5] + \frac{1}{\log_2(1 + \text{rank}_{q'_2})} \cdot [\text{rank}_{q'_2} \leq 5] + \frac{1}{\log_2(1 + \text{rank}_{q'_3})} \cdot [\text{rank}_{q'_3} \leq 5] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 5 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 5 $ → $ 1 $.

Сумма:
$$
\text{DCG@5} = \frac{1}{3} \cdot (0.631 + 0.387 + 1) = \frac{1}{3} \cdot 2.018 \approx 0.673.
$$

$$
\boxed{\text{DCG@5} \approx 0.673}.
$$

#### ***Вопрос 4:***
* Найдите максимум `Hits@47 - DCG@1`?



Будем рассматривать на примере одного вопроса.

Для всех позиций, кроме 1, DCG@1 = 0. Для позиции 1 DCG@1 = 1.
Для позиции 1 Hits@47 = 1. Тогда выражение равно 1-1 = 0.

Для позиций 2-47 Hits@47 = 1, а DCG@1 = 0. Тогда выражение равно 1.

После 48 позиции включительно выражение будет равняться 0 - 0 = 0.

Таким образом, максимум равен 1 и достигается при позиции 2-47.

Второй вариант решения - кодом (учтем, что после 47й позиции никаких изменений в значениях нет).

In [ ]:
def compute_dcg(pos):
    return int(pos <= 1) * 1/np.log2(1+pos)

def compute_hits(pos):
    return int(pos <= 47)

max_val = compute_hits(1) - compute_dcg(1)
for pos in range(2, 50):
    max_val = max(max_val, compute_hits(pos) - compute_dcg(pos))

max_val

np.float64(1.0)

### HITS\_COUNT и DCG\_SCORE

Каждая функция имеет два аргумента: $dup\_ranks$ и $k$.

$dup\_ranks$ является списком, который содержит рейтинги дубликатов (их позиции в ранжированном списке).

К примеру для <font color='red'>"Что такое язык python?"</font> $dup\_ranks = [2]$.

In [ ]:
def hits_count(dup_ranks, k):
    """
        dup_ranks: list индексов дубликатов
        k: пороговое значение для ранга
        result: вернуть Hits@k
    """
    # Подсчитываем количество дубликатов, чей ранг <= k
    hits_value = sum([rank <= k for rank in dup_ranks]) / len(dup_ranks)
    return hits_value

In [ ]:
dup_ranks = [2]

k = 1
hits_value = hits_count(dup_ranks, k)
print(f"Hits@1 = {hits_value}")

k = 4
hits_value = hits_count(dup_ranks, k)
print(f"Hits@4 = {hits_value}")

Hits@1 = 0.0
Hits@4 = 1.0


In [ ]:
import math

def dcg_score(dup_ranks, k):
    """
        dup_ranks: list индексов дубликатов
        k: пороговое значение для ранга
        result: вернуть DCG@k
    """
    # Вычисляем сумму для всех релевантных дубликатов
    dcg_sum = sum([int(rank <= k)/np.log2(1+rank) for rank in dup_ranks])

    # Делим на общее количество вопросов
    dcg_value = dcg_sum / len(dup_ranks)
    return dcg_value

In [ ]:
# Пример списка позиций дубликатов
dup_ranks = [2]

# Вычисляем DCG@1
dcg_value = dcg_score(dup_ranks, k=1)
print(f"DCG@1 = {dcg_value:.3f}")

# Вычисляем DCG@4
dcg_value = dcg_score(dup_ranks, k=4)
print(f"DCG@10 = {dcg_value:.3f}")

DCG@1 = 0.000
DCG@10 = 0.631


Протестируем функции. Пусть $N = 1$, то есть один эксперимент. Будем искать копию вопроса и оценивать метрики.

In [ ]:
import pandas as pd

In [ ]:
copy_answers = ["How does the catch keyword determine the type of exception that was thrown",]

# наши кандидаты
candidates_ranking = [["How Can I Make These Links Rotate in PHP",
                       "How does the catch keyword determine the type of exception that was thrown",
                       "NSLog array description not memory address",
                       "PECL_HTTP not recognised php ubuntu"],]

# dup_ranks — позиции наших копий, так как эксперимент один, то этот массив длины 1
dup_ranks = [ar.index(question)+1 for ar, question in zip(candidates_ranking, copy_answers)]

# вычисляем метрику для разных k
print('Ваш ответ HIT:', [hits_count(dup_ranks, k) for k in range(1, 5)])
print('Ваш ответ DCG:', [round(dcg_score(dup_ranks, k), 5) for k in range(1, 5)])

Ваш ответ HIT: [0.0, 1.0, 1.0, 1.0]
Ваш ответ DCG: [np.float64(0.0), np.float64(0.63093), np.float64(0.63093), np.float64(0.63093)]


У вас должно получиться

In [ ]:
# correct_answers - метрика для разных k
correct_answers = pd.DataFrame([[0, 1, 1, 1], [0, 1 / (np.log2(3)), 1 / (np.log2(3)), 1 / (np.log2(3))]],
                               index=['HITS', 'DCG'], columns=range(1,5))
correct_answers

,1,2,3,4
HITS,0,1.00000,1.00000,1.00000
DCG,0,0.63093,0.63093,0.63093


### Данные
[arxiv link](https://drive.google.com/file/d/1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_/edit)

`train.tsv` - выборка для обучения.<br> В каждой строке через табуляцию записаны: **<вопрос>, <похожий вопрос>**

`validation.tsv` - тестовая выборка.<br> В каждой строке через табуляцию записаны: **<вопрос>, <похожий вопрос>, <отрицательный пример 1>, <отрицательный пример 2>, ...**

In [ ]:
import gdown
fid = '1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_'
!gdown --id 1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_
From (redirected): https://drive.google.com/uc?id=1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_&confirm=t&uuid=e9f8d304-bb85-4ab0-aefa-0a476927d38f
To: /content/stackoverflow_similar_questions.zip
100% 131M/131M [00:00<00:00, 142MB/s]


In [ ]:
!unzip stackoverflow_similar_questions.zip

Archive:  stackoverflow_similar_questions.zip
   creating: data/
  inflating: data/.DS_Store          
   creating: __MACOSX/
   creating: __MACOSX/data/
  inflating: __MACOSX/data/._.DS_Store  
  inflating: data/train.tsv          
  inflating: data/validation.tsv     


Считайте данные.

In [ ]:
def read_corpus(filename):
    data = []
    with open(filename, encoding='utf-8') as file:
        for line in file:
            data.append(line.strip().split('\t'))
    return data

Нам понадобиться только файл validation.

In [ ]:
validation_data = read_corpus('./data/validation.tsv')

Кол-во строк

In [ ]:
len(validation_data)

3760

Размер нескольких первых строк

In [ ]:
for i in range(25):
    print(i + 1, len(validation_data[i]))

1 1001
2 1001
3 1001
4 1001
5 1001
6 1001
7 1001
8 1001
9 1001
10 1001
11 1001
12 1001
13 1001
14 1001
15 1001
16 1001
17 1001
18 1001
19 1001
20 1001
21 1001
22 1001
23 1001
24 1001
25 1001


### Ранжирование без обучения

Реализуйте функцию ранжирования кандидатов на основе косинусного расстояния. Функция должна по списку кандидатов вернуть отсортированный список пар (позиция в исходном списке кандидатов, кандидат). При этом позиция кандидата в полученном списке является его рейтингом (первый - лучший). Например, если исходный список кандидатов был [a, b, c], и самый похожий на исходный вопрос среди них - c, затем a, и в конце b, то функция должна вернуть список **[(2, c), (0, a), (1, b)]**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy

In [ ]:
def cosine_distance(a, b):
    return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
def rank_candidates(question, candidates, embeddings, tokenizer, dim=200):
    """
        question: строка
        candidates: массив строк(кандидатов) [a, b, c]
        result: пары (начальная позиция, кандидат) [(2, c), (0, a), (1, b)]
    """
    question_embed = question_to_vec(question, embeddings, tokenizer)
    data_embed = [question_to_vec(data, embeddings, tokenizer) for data in candidates]
    result = []
    for i, (vec, data) in enumerate(zip(data_embed, candidates)):
        cos_dist = cosine_distance(question_embed, vec)
        result.append((cos_dist, i, data))
    result.sort(key = lambda x: x[0])
    return [t[1:] for t in result]

Протестируйте работу функции на примерах ниже. Пусть $N=2$, то есть два эксперимента

In [ ]:
questions = ['converting string to list', 'Sending array via Ajax fails']

candidates = [['Convert Google results object (pure js) to Python object', # первый эксперимент
               'C# create cookie from string and send it',
               'How to use jQuery AJAX for an outside domain?'],

              ['Getting all list items of an unordered list in PHP',      # второй эксперимент
               'WPF- How to update the changes in list item of a list',
               'select2 not displaying search results']]

In [51]:
for question, q_candidates in zip(questions, candidates):
        ranks = rank_candidates(question.lower(), q_candidates, wv_embeddings, tokenizer)
        print(ranks)
        print()

[(1, 'C# create cookie from string and send it'), (0, 'Convert Google results object (pure js) to Python object'), (2, 'How to use jQuery AJAX for an outside domain?')]

[(1, 'WPF- How to update the changes in list item of a list'), (2, 'select2 not displaying search results'), (0, 'Getting all list items of an unordered list in PHP')]



Для первого экперимента вы можете полностью сравнить ваши ответы и правильные ответы. Но для второго эксперимента два ответа на кандидаты будут <b>скрыты</b>(*)

In [ ]:
# должно вывести
results = [[(1, 'C# create cookie from string and send it'),
            (0, 'Convert Google results object (pure js) to Python object'),
            (2, 'How to use jQuery AJAX for an outside domain?')],
           [('*', 'Getting all list items of an unordered list in PHP'), #скрыт
            ('*', 'select2 not displaying search results'), #скрыт
            ('*', 'WPF- How to update the changes in list item of a list')]] #скрыт

Последовательность начальных индексов вы должны получить `для эксперимента 1`  1, 0, 2.

#### ***Вопрос 5:***
* Какую последовательность начальных индексов вы получили `для эксперимента 2`(перечисление без запятой и пробелов, например, `102` для первого эксперимента?


**102**

Теперь мы можем оценить качество нашего метода. Запустите следующие два блока кода для получения результата. Обратите внимание, что вычисление расстояния между векторами занимает некоторое время (примерно 10 минут). Можете взять для validation 1000 примеров.

In [ ]:
from tqdm.notebook import tqdm

In [52]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    q, *ex = line
    ranks = rank_candidates(q.lower(), ex, wv_embeddings, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_4207/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


In [53]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.538 | Hits@   1: 0.538
DCG@   5: 0.637 | Hits@   5: 0.721
DCG@  10: 0.658 | Hits@  10: 0.787
DCG@ 100: 0.702 | Hits@ 100: 0.991
DCG@ 500: 0.703 | Hits@ 500: 1.000
DCG@1000: 0.703 | Hits@1000: 1.000


Из формул выше можно понять, что

- $ \text{Hits@K} $ **монотонно неубывающая функция** $ K $, которая стремится к 1 при $ K \to \infty $.

- $ \text{DCG@K} $ **монотонно неубывающая функция** $ K $, но рост замедляется с увеличением $ K $ из-за убывания веса $ \frac{1}{\log_2(1 + \text{rank}_{q'_i})} $.

### Эмбеддинги, обученные на корпусе похожих вопросов

In [ ]:
train_data = read_corpus('./data/train.tsv')

Улучшите качество модели.<br>Склеим вопросы в пары и обучим на них модель Word2Vec из gensim. Выберите размер window. Объясните свой выбор.

***Рассмотрим подробнее*** данное склеивание.

1. Каждая строка из train_data разбивается на вопрос (question) и список кандидатов.

2. Для каждого кандидата вопрос склеивается с ним в одну строку.

3. Склеенная строка (combined_text) токенизируется, и полученный список токенов добавляется в общий корпус (corpus).

***Пример***

    Вопрос: "What is Python?"
    Кандидаты: ["Python is a programming language", "Java is another language"]
    Склеенные строки:
        "What is Python? Python is a programming language"
        "What is Python? Java is another language"
         
    Токенизированные списки:
        ['what', 'is', 'python', 'python', 'is', 'a', 'programming', 'language']
        ['what', 'is', 'python', 'java', 'is', 'another', 'language']
         
     

In [ ]:
train_data[111258]

['Determine if the device is a smartphone or tablet?',
 'Change imageView params in all cards together']

In [ ]:
from gensim.models import Word2Vec

## Наивный w2v

Сначала опробуем word2vec, просто подбирая гиперпараметры. Спойлер - качества 0.519 достичь не удалось

In [ ]:
def validate(embeddings_trained):
    wv_ranking = []
    for i, line in enumerate(tqdm(validation_data)):
        q, *ex = line
        ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
        wv_ranking.append([r[0] for r in ranks].index(0) + 1)
    return wv_ranking

In [ ]:
def try_word2vec(pre_processed_text, tokenizer, window_size=7, min_count=3, epochs=1):
    corpus = [tokenizer.tokenize(f"{i[0]} {i[1]}".lower()) for i in pre_processed_text]
    embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=window_size,                # Размер окна контекста
    min_count=min_count,
    sg=1,
    epochs=epochs,
    workers=16                # Количество потоков
    ).wv
    return validate(embeddings_trained)

Посмотрим на гиперпараметры - размер окна и min_count слов. Окно должно захватывать оба вопроса, поэтому минимальный размер рассматриваемого окна равен 5.

In [ ]:
for window_size in range(5, 8):
    for min_count in [3, 5, 7, 10]:
        print(window_size, min_count)
        ranks = try_word2vec(train_data, MyTokenizer(), window_size, min_count, epochs=20)

        for k in tqdm([1, 5, 10, 100, 500, 1000]):
            print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

5 3


  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_7244/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.427 | Hits@   1: 0.427
DCG@   5: 0.506 | Hits@   5: 0.574
DCG@  10: 0.528 | Hits@  10: 0.643
DCG@ 100: 0.587 | Hits@ 100: 0.938
DCG@ 500: 0.596 | Hits@ 500: 1.000
DCG@1000: 0.596 | Hits@1000: 1.000
5 5


...

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.437 | Hits@   1: 0.437
DCG@   5: 0.517 | Hits@   5: 0.588
DCG@  10: 0.539 | Hits@  10: 0.654
DCG@ 100: 0.597 | Hits@ 100: 0.941
DCG@ 500: 0.605 | Hits@ 500: 1.000
DCG@1000: 0.605 | Hits@1000: 1.000
7 10


  0%|          | 0/3760 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.437 | Hits@   1: 0.437
DCG@   5: 0.517 | Hits@   5: 0.586
DCG@  10: 0.541 | Hits@  10: 0.660
DCG@ 100: 0.598 | Hits@ 100: 0.941
DCG@ 500: 0.606 | Hits@ 500: 1.000
DCG@1000: 0.606 | Hits@1000: 1.000


Видим, что метрики не очень зависят от этих гиперпараметров. Зафиксируем 6, 10.

Посмотрим, как влияет число эпох:

In [ ]:
for ep in [1, 2, 5, 10, 15, 20, 30, 40]:
    print(f"epochs: {ep}")
    ranks = try_word2vec(train_data, MyTokenizer(), 6, 10, epochs=ep)

    for k in tqdm([1, 5, 10, 100, 500, 1000]):
        print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

epochs: 1


  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_7244/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.386 | Hits@   1: 0.386
DCG@   5: 0.468 | Hits@   5: 0.540
DCG@  10: 0.489 | Hits@  10: 0.606
DCG@ 100: 0.552 | Hits@ 100: 0.919
DCG@ 500: 0.563 | Hits@ 500: 1.000
DCG@1000: 0.563 | Hits@1000: 1.000
epochs: 2


...

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.434 | Hits@   1: 0.434
DCG@   5: 0.516 | Hits@   5: 0.588
DCG@  10: 0.540 | Hits@  10: 0.661
DCG@ 100: 0.597 | Hits@ 100: 0.944
DCG@ 500: 0.605 | Hits@ 500: 1.000
DCG@1000: 0.605 | Hits@1000: 1.000
epochs: 40


  0%|          | 0/3760 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.433 | Hits@   1: 0.433
DCG@   5: 0.514 | Hits@   5: 0.584
DCG@  10: 0.539 | Hits@  10: 0.660
DCG@ 100: 0.595 | Hits@ 100: 0.940
DCG@ 500: 0.604 | Hits@ 500: 1.000
DCG@1000: 0.604 | Hits@1000: 1.000


Видим, что мы выходим на плато довольно скоро. Размер эпох зафиксируем на 15.

## Стемминг

Попробуем привести слова к +- одной форме с помощью их "обрезки" до корня - таким образом, мы уменьшим влияние различных словоформ, например plugins и plugin будут кодироваться одним вектором.

In [ ]:
from nltk.stem import PorterStemmer

def build_embeddings_stemming(pre_processed_text, tokenizer):
    stemmer = PorterStemmer()
    def tokenize_and_stem(text):
        #tokens = [tokenizer.tokenize(i) for i in pre_processed_text]
        tokens = tokenizer.tokenize(f"{text[0]} {text[1]}".lower())
        return [stemmer.stem(t) for t in tokens]

    corpus = [tokenize_and_stem(i) for i in pre_processed_text]
    return corpus


def try_word2vec_stem(pre_processed_text, tokenizer, window_size, min_count, vector_size=200, epochs=15):
    corpus = build_embeddings_stemming(pre_processed_text, tokenizer)
    print("Starting word2vec")
    embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=vector_size,         # Размерность векторов
    window=window_size,                # Размер окна контекста
    min_count=min_count,
    sg = 1,
    epochs=epochs, # Минимальная частота слов
    workers=16              # Количество потоков
    ).wv
    return validate(embeddings_trained)

Здесь сравним результаты на 2х токенайзерах.

In [ ]:
ranks = try_word2vec_stem(train_data, MyTokenizer(), 6, 10)

for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

Starting word2vec


  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_7244/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.394 | Hits@   1: 0.394
DCG@   5: 0.460 | Hits@   5: 0.519
DCG@  10: 0.480 | Hits@  10: 0.583
DCG@ 100: 0.554 | Hits@ 100: 0.948
DCG@ 500: 0.561 | Hits@ 500: 1.000
DCG@1000: 0.561 | Hits@1000: 1.000


In [ ]:
ranks = try_word2vec_stem(train_data, WordPunctTokenizer(), 6, 10)

for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

Starting word2vec


  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_7244/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.328 | Hits@   1: 0.328
DCG@   5: 0.396 | Hits@   5: 0.457
DCG@  10: 0.417 | Hits@  10: 0.522
DCG@ 100: 0.490 | Hits@ 100: 0.894
DCG@ 500: 0.505 | Hits@ 500: 1.000
DCG@1000: 0.505 | Hits@1000: 1.000


My_Tokenizer оставляет только слова, в то время как WordPunctTokenizer оставляет еще и пунктуацию - ( и ! тоже станут токенами, что не очень информативно и "засоряет" окно.

## Биграмы

Соберем биграмы - так, machine learning перейдет в один токен - machine_learning.

In [ ]:
from gensim.models.phrases import Phrases, Phraser

def try_word2vec_bigram(pre_processed_text, tokenizer, window_size=7, min_count=3, epochs=15):
    corpus = [tokenizer.tokenize(f"{i[0]} {i[1]}".lower()) for i in pre_processed_text]

    phrases = Phrases(corpus, min_count=5, threshold=10)
    bigram = Phraser(phrases)

    corpus_bigram = [bigram[sent] for sent in corpus]

    embeddings_trained = Word2Vec(
        sentences=corpus_bigram,
        vector_size=200,
        window=window_size,
        min_count=min_count,
        sg=1,
        epochs=epochs,
        workers=16
    ).wv

    return validate_bigram(embeddings_trained, tokenizer, bigram)


def validate_bigram(embeddings_trained, tokenizer, bigram, dim=200):
    wv_ranking = []

    for line in tqdm(validation_data):
        q, *candidates = line

        q_tokens = bigram[tokenizer.tokenize(q.lower())]
        q_vecs = [embeddings_trained[w] for w in q_tokens if w in embeddings_trained]
        if q_vecs == []:
            question_embed = np.zeros(dim)
        else:
            question_embed = np.mean(q_vecs, axis=0)

        result = []

        for i, cand in enumerate(candidates):
            tokens = bigram[tokenizer.tokenize(cand.lower())]
            vecs = [embeddings_trained[w] for w in tokens if w in embeddings_trained]

            if vecs == []:
                cand_vec = np.zeros(dim)
            else:
                cand_vec = np.mean(vecs, axis=0)

            cos_dist = cosine_distance(question_embed, cand_vec)
            result.append((cos_dist, i))

        result.sort(key=lambda x: x[0])
        ranks = [r[1] for r in result]

        wv_ranking.append(ranks.index(0) + 1)

    return wv_ranking

Здесь представлены результаты на 15 и 25 эпохах.

In [ ]:
ranks = try_word2vec_bigram(train_data, MyTokenizer(), 6, 10)

for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_7244/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.457 | Hits@   1: 0.457
DCG@   5: 0.549 | Hits@   5: 0.628
DCG@  10: 0.567 | Hits@  10: 0.685
DCG@ 100: 0.602 | Hits@ 100: 0.854
DCG@ 500: 0.616 | Hits@ 500: 0.962
DCG@1000: 0.620 | Hits@1000: 1.000


In [ ]:
ranks = try_word2vec_bigram(train_data, MyTokenizer(), 6, 10, 25)

for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_7244/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.470 | Hits@   1: 0.470
DCG@   5: 0.560 | Hits@   5: 0.639
DCG@  10: 0.580 | Hits@  10: 0.699
DCG@ 100: 0.614 | Hits@ 100: 0.862
DCG@ 500: 0.627 | Hits@ 500: 0.966
DCG@1000: 0.631 | Hits@1000: 1.000


Увеличение числа эпох до 25 дает прибавку к качеству, но не критичную, всего на несколько процентов.

## Лемматизация

Лемматизация позволяет также предподготовить наш датасет, приводя слова к некоторой начальной форме. Здесь я использую spacy.

In [ ]:
import spacy
def lemmatize_tokens(tokens):
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(" ".join(tokens))
    return [t.lemma_ for t in doc]

In [ ]:
def try_word2vec_lemm(pre_processed_text, tokenizer, window_size, min_count, epochs):
    corpus = [lemmatize_tokens(tokenizer.tokenize(f"{i[0]} {i[1]}".lower())) for i in pre_processed_text]
    print("Starting word2vec")
    embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=window_size,                # Размер окна контекста
    min_count=min_count,
    sg = 1,
    epochs = epochs,# Минимальная частота слов
    workers=16              # Количество потоков
    ).wv
    return validate(embeddings_trained)

К сожалению, лемматизация занимает очень много времени, поэтому я смогла показать примеры только на очень малом числе данных.

In [ ]:
ranks = try_word2vec_lemm(train_data[:1_000], WordPunctTokenizer(), 6, 10, 25)

for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(ranks, k), k, hits_count(ranks, k)))

Starting word2vec


  0%|          | 0/3760 [00:00<?, ?it/s]

/tmp/ipykernel_4207/2447080678.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.216 | Hits@   1: 0.216
DCG@   5: 0.281 | Hits@   5: 0.341
DCG@  10: 0.306 | Hits@  10: 0.420
DCG@ 100: 0.402 | Hits@ 100: 0.903
DCG@ 500: 0.415 | Hits@ 500: 1.000
DCG@1000: 0.415 | Hits@1000: 1.000


### Замечание:
Решить эту задачу с помощью обучения полноценной нейронной сети будет вам предложено, как часть задания в одной из домашних работ по теме "Диалоговые системы".

Напишите свой вывод о полученных результатах.
* Какой принцип токенизации даёт качество лучше и почему?
* Помогает ли нормализация слов?
* Какие эмбеддинги лучше справляются с задачей и почему?
* Почему получилось плохое качество решения задачи?
* Предложите свой подход к решению задачи.

## Вывод:



*   Во многих подходах 2 рассмотренных метода токенизации - по словам и по словам + по пунктуации давали сопоставимые результаты. В других - токенизация по словам давала лучшие результаты. Это связано с тем, что получающиеся из знаков пунктуации токены не несут сильной нагрузки (как например, смайлики для предсказания настроения твита), но загружают контекст.

*   Нормализация слов значительно повышает качество за счет приведения схожих слов к одному значению, следовательно, к одному вектору. Проверить лемматизацию в полном объеме, к сожалению, не удалось.

*   В данном случае эмбеддинги, полученные в ходе работы, не превзошли предобученных эмбеддингов. Hits@1 0.470 против 0.538.

Чем отличаются эмбеддинги и что могло испортить обучение:

* Неправильный размер окна (в статье - 5)
* Неправильная предподготовка - не удалялись стоп-слова, потенциально удалялись важные символы (а надо различать C++, C, C#)
* В десятки раз меньше данных для обучения

Что можно улучшить

* Изменить правила получения векторов предложений - использовать не среднее, вместо нулевого вектора взять случайный шум
* Использовать лемматизацию
* Почистить текст от "мусорных" слов
* fine-tune на размер вектора - мной взят использованный в статье - 200.